# First Implementation of the RSA model 

In [1]:
import numpy as np 
import random as rd
import spacy
from sklearn.metrics.pairwise import cosine_similarity
import wordfreq
import random as rd
import json 

nlp = spacy.load('de_core_news_sm')


In [2]:
def get_wlist(pos="NOUN", n=100, from_n = 10000, savepath="../output/random_nouns.json"):
    frequent_words = nlp(" ".join(wordfreq.top_n_list("de", from_n)))
    frequent_nouns = [w.lemma_ for w in frequent_words if (w.pos_==pos and w.lemma_[0].isupper())]
    rd_n = rd.sample(frequent_nouns, n)
    if savepath:
        with open("../output/random_nouns.json", "w") as final:
            json.dump(rd_n, final)
    return(rd_n)

V = get_wlist()[:10]


In [3]:
print(V)

['Opel', 'Kriege', 'Weg', 'Bündnis', 'Würdest', 'Einblick', 'Streik', 'Erfolge', 'Christ', 'Hand']


In [67]:
import itertools 

freq = {w: wordfreq.word_frequency(w, "de") for w in V}
players = ["P1", "P2", "P3"]
roles = {"P1": "agent", "P2": "agent", "P3": "imposter"}
target = rd.sample(V, 1)[0]
target_index = [i for i, t in enumerate(V) if t==target][0]
print("target_index:", target_index)
rationality = {"P1": 3, "P2": 3, "P3": 5}
C = {k: 1 for k in V}


target_index: 6


In [34]:
print(target)

Opel


In [35]:
W = []
imposter="P3"

for p in players:
    for w in V:
        W.append((p,w))

print(W)

[('P1', 'Opel'), ('P1', 'Kriege'), ('P1', 'Weg'), ('P1', 'Bündnis'), ('P1', 'Würdest'), ('P1', 'Einblick'), ('P1', 'Streik'), ('P1', 'Erfolge'), ('P1', 'Christ'), ('P1', 'Hand'), ('P2', 'Opel'), ('P2', 'Kriege'), ('P2', 'Weg'), ('P2', 'Bündnis'), ('P2', 'Würdest'), ('P2', 'Einblick'), ('P2', 'Streik'), ('P2', 'Erfolge'), ('P2', 'Christ'), ('P2', 'Hand'), ('P3', 'Opel'), ('P3', 'Kriege'), ('P3', 'Weg'), ('P3', 'Bündnis'), ('P3', 'Würdest'), ('P3', 'Einblick'), ('P3', 'Streik'), ('P3', 'Erfolge'), ('P3', 'Christ'), ('P3', 'Hand')]


In [ ]:
priors_P1 = []
# P1 weiß wer der Imposter ist 
for p, w in W: 
    if p in ["P1", "P2"]:
        priors_P1.append(0)
    elif w==target:
        priors_P1.append(1)
    else:
        priors_P1.append(0)

print(priors_P1)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [37]:
priors_P1_norm = [w/sum(priors_P1) for w in priors_P1]
print(priors_P1_norm)

[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [38]:
# modelled priors for P2, who is also an agent (with role of P2 being known), but that doesn't know the roles of the others 
priors_P2 = []
for p, w in W: 
    if p=="P2": 
        priors_P2.append(0)
    elif w==target:
        priors_P2.append(0.5)
    else:
        priors_P2.append(0)

priors_P2_norm = [w/sum(priors_P2) for w in priors_P2]
print(priors_P2_norm)


[0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.5, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]


In [10]:
# modelled priors for P3, who is an imposter (with role of P3 being known)
priors_P3 = []
for p, t in W: 
    if p=="P3": 
        priors_P3.append(freq[t])
    else:
        priors_P3.append(0)

priors_P3_norm = [w/sum(priors_P3) for w in priors_P3]
print(priors_P3_norm)


[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.016621299352010215, 0.03721725724471852, 0.5636788475899116, 0.018187073928648857, 0.05142967263266929, 0.016621299352010215, 0.00934646978055067, 0.033965263893238264, 0.012044419820297258, 0.24088839640594514]


In [45]:
priors_P3 = np.array(priors_P3_norm)
priors_P3.shape = (3,10)
print(priors_P3)

[[0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.        ]
 [0.0166213  0.03721726 0.56367885 0.01818707 0.05142967 0.0166213
  0.00934647 0.03396526 0.01204442 0.2408884 ]]


In [39]:
def get_similarity(u, t): 
    u_vec = nlp(u).vector
    t_vec = nlp(t).vector
    return(cosine_similarity([u_vec], [t_vec])[0][0])

def get_sim_table(V, t ): 
    # dim: m x n, so for each of the m target words there are similarities for each of the n words 
    sim_vec = []
    for t in V: 
        vec_t = []
        for w in V: 
            if t==w: 
                vec_t.append(np.nan)
            else:
                vec_t.append(1 - get_similarity(w,t))
        vec_sum = np.nansum(vec_t)
        vec_t = [vec/vec_sum for vec in vec_t]
        sim_vec.append(vec_t)
    return(np.array(sim_vec))

sim_table = get_sim_table(V, target)
print(sim_table)

[[       nan 0.10527052 0.10527434 0.15089101 0.09472178 0.10522482
  0.09914713 0.12065966 0.07671133 0.14209941]
 [0.11655159        nan 0.10532193 0.18939952 0.12417051 0.09036577
  0.08675872 0.04830778 0.11603882 0.12308536]
 [0.12764011 0.1153379         nan 0.18806868 0.10009492 0.0568062
  0.07929278 0.13114111 0.0845474  0.11707091]
 [0.11769335 0.13343078 0.12098748        nan 0.08720186 0.10989319
  0.12332111 0.12611012 0.08578461 0.09557749]
 [0.10365206 0.12272559 0.09033904 0.12233909        nan 0.09401613
  0.1263637  0.1446531  0.06655694 0.12935435]
 [0.14118346 0.1095111  0.06286323 0.18903744 0.11527627        nan
  0.06345439 0.11071217 0.07169552 0.13626641]
 [0.11594828 0.09164018 0.07648093 0.18489836 0.13504498 0.05530702
         nan 0.12665187 0.0888951  0.12513328]
 [0.11613596 0.04199623 0.10410673 0.15562021 0.12723432 0.07942082
  0.10423943        nan 0.13368755 0.13755876]
 [0.09803347 0.13393892 0.08911494 0.14055157 0.07772849 0.06828752
  0.09714233 

In [42]:
sim_table[target_index,3]

np.float64(0.1508910061898415)

In [41]:
print(priors_P2)#[:,:,None])

[0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0.5, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [50]:
target_imp_ind = np.argmax(priors_P3[2])
print(target_imp_ind)
target_imp = V[target_imp_ind]

2


In [54]:
def get_P_u_w(u,w,t_imp,speaker="P1"):
    imp_ind, t_ind = w

    if players[imp_ind]==speaker: 
        P_u_w = sim_table[t_ind, u]
    else:
        P_u_w = sim_table[t_imp, u]
    return(P_u_w)

In [57]:
u = V[5]
u_ind = V.index(u)
print("u:", u)
priors_P2 = np.array(priors_P2)
priors_P2.shape = (3,10)
#print(priors_P2)
# Ähnlichkeit von u und verschiedenen Targetwörtern x jeweilige Priors für P1 
L0_P2_u = sim_table[:,3] * priors_P2#[:,:,None]

W_inds = [(players.index(p),V.index(t)) for p,t in W]

l0_P2_u_P1_t = sim_table[target_index, 3] * priors_P2[0,target_index]
print("l0_P2:", l0_P2_u_P1_t)
SUM = np.nansum([get_P_u_w(u_ind,w,target_imp_ind) * priors_P2[w[0],w[1]] for w in W_inds])
print("sum:", SUM)
L0_P2_u_P1_t = l0_P2_u_P1_t/SUM
print(L0_P2_u_P1_t)
#print(L0_P2_u_P1_t.shape)
# das sieht schon irgendwie sinnvoll aus, aber ich checke es nicht
# v Größe 10 x 10, also Zahl der Targets mal Zahl der möglichen Aussagen 
# jetzt muss normalisiert werden (auch wenn das in diesem Fall nix ändert) - für jedes w muss über alle Ts normalisiert werden 

u: Einblick
l0_P2: 0.07544550309492075
sum: 0.08101551215913279
0.9312476226371157


In [81]:
priors = {"P1": priors_P1, "P2": priors_P2, "P3": priors_P3}

def listen(u, w, listener="P2", speaker="P1"):
    imp, t = w 
    imp = players.index(imp)
    priors_l = priors[listener]
    t = V.index(t)
    s = players.index(speaker)
    l = players.index(listener)
    u = V.index(u)
    P_w_u = sim_table[t, u] * priors_l[imp,t]
    SUM = np.nansum([get_P_u_w(u_ind,w,target_imp_ind) * priors_l[w[0],w[1]] for w in W_inds])
    P_w_u = P_w_u/SUM
    return(P_w_u)

In [117]:
def utility(u, w, partner="P2", imp="P3"):
    #print(f"listen(u,w,listener=partner): {listen(u,w,listener=partner)}, listen(u,(P3,t),listener=imp): {listen(u,(imp,w[1]),listener=imp)}")
    # Anwendung von log-Funktion auf Differenz beruht ausschließlich auf ausprobieren
    U = np.log(listen(u,w,listener=partner)) + np.log(listen(u,(imp,w[1]),listener=imp)) # weil auf logarithmischer Skala negativ 
    #SUM = np.nansum([np.log(listen(u_n,w,listener=partner)) - np.log(listen(u_n,w,listener=imp)) for u_n in range(len(V))])
    #print(f"U({u},{w}): {U}")
    return(U)

In [ ]:
alpha = 5
u = V[9]
# third person is imposter, first word is target word 
w = ("P3",V[0])
print("w:",w)
print(f"U(u,w): {utility(u,w)}")
P_u_w = np.exp(alpha * utility(u,w))
print(f"P(u|w) before normalisation: {P_u_w}")
print("all speaker probs:", [np.exp(alpha * utility(u_n, w)) for u_n in V])
SUM = np.nansum([np.exp(alpha * utility(u_n, w)) for u_n in V])
print(f"sum: {SUM}")
P_u_w = P_u_w/SUM
print(P_u_w)



w: ('P3', 'Opel')
U(Hand,('P3', 'Opel')): -3.3114498372466543
U(u,w): -3.3114498372466543
U(Hand,('P3', 'Opel')): -3.3114498372466543
P(u|w) before normalisation: 6.445818059872121e-08
U(Opel,('P3', 'Opel')): nan
U(Kriege,('P3', 'Opel')): -3.9114367230915854
U(Weg,('P3', 'Opel')): -3.911364270435375
U(Bündnis,('P3', 'Opel')): -3.191388087558018
U(Würdest,('P3', 'Opel')): -4.122615584152155
U(Einblick,('P3', 'Opel')): -3.91230512420648
U(Streik,('P3', 'Opel')): -4.031293847980054
U(Erfolge,('P3', 'Opel')): -3.638555884697547
U(Christ,('P3', 'Opel')): -4.544404883662885
U(Hand,('P3', 'Opel')): -3.3114498372466543
all speaker probs: [np.float64(nan), np.float64(3.209394279908687e-09), np.float64(3.210557136228069e-09), np.float64(1.174867309540028e-07), np.float64(1.1164878568864987e-09), np.float64(3.1954892816904406e-09), np.float64(1.7626116426507822e-09), np.float64(1.2559613816315045e-08), np.float64(1.355033953059251e-10), np.float64(6.445818059872121e-08)]
U(Opel,('P3', 'Opel')): n

In [116]:
def S1(u,w, alpha=1): 
    P_u_w = np.exp(alpha * utility(u,w))
    SUM = np.nansum([np.exp(alpha * utility(u_n, w)) for u_n in V])
    P_u_w = P_u_w/SUM
    return(P_u_w)


u_probs = []
for u in V:
    prob = S1(u,w)
    u_probs.append(prob)

print(u_probs)
print(np.nansum(u_probs))

U(Opel,('P3', 'Opel')): nan
U(Opel,('P3', 'Opel')): nan
U(Kriege,('P3', 'Opel')): -3.9114367230915854
U(Weg,('P3', 'Opel')): -3.911364270435375
U(Bündnis,('P3', 'Opel')): -3.191388087558018
U(Würdest,('P3', 'Opel')): -4.122615584152155
U(Einblick,('P3', 'Opel')): -3.91230512420648
U(Streik,('P3', 'Opel')): -4.031293847980054
U(Erfolge,('P3', 'Opel')): -3.638555884697547
U(Christ,('P3', 'Opel')): -4.544404883662885
U(Hand,('P3', 'Opel')): -3.3114498372466543
U(Kriege,('P3', 'Opel')): -3.9114367230915854
U(Opel,('P3', 'Opel')): nan
U(Kriege,('P3', 'Opel')): -3.9114367230915854
U(Weg,('P3', 'Opel')): -3.911364270435375
U(Bündnis,('P3', 'Opel')): -3.191388087558018
U(Würdest,('P3', 'Opel')): -4.122615584152155
U(Einblick,('P3', 'Opel')): -3.91230512420648
U(Streik,('P3', 'Opel')): -4.031293847980054
U(Erfolge,('P3', 'Opel')): -3.638555884697547
U(Christ,('P3', 'Opel')): -4.544404883662885
U(Hand,('P3', 'Opel')): -3.3114498372466543
U(Weg,('P3', 'Opel')): -3.911364270435375
U(Opel,('P3', 'O

In [107]:
np.exp(np.log(0.8)+np.log(0.3))

np.float64(0.24)

In [95]:
np.log(0.61) - 0.8 * np.log(0.03)

np.float64(2.3109499960412054)

In [78]:
t = V[0]
u = V[7]
print(f"u: {u}, t: {t}")
w = ("P1", t)
listen(u,w)

u: Erfolge, t: Opel


np.float64(0.7446701124608864)

In [ ]:
# Load a large English model with word vectors included

# Access the vector for a specific word

def get_similarity(u, t): 
    u_vec = nlp(u).vector
    t_vec = nlp(t).vector
    return(cosine_similarity([u_vec], [t_vec])[0][0])

def get_sim_table(V, t ): 
    # dim: m x n, so for each of the m target words there are similarities for each of the n words 
    sim_vec = []
    for t in V: 
        vec_t = []
        for w in V: 
            if t==w: 
                vec_t.append(0)
            vec_t.append(get_similarity(w,t))
        sim_vec.append(vec_t)
    return(np.array(sim_vec))

sim_table = get_sim_table(V, target)

class Agent:
    def __init__(self, player): 
         self.player = player
         self.role = roles[player]
         self.target = target
         self.target_imp = self.get_target()
         #self.P_w = {p: [2/3, 1/3] for p in players}
         self.alpha = rationality[player]
         self.sims = {w: get_similarity(w,self.target) * freq[w] for w in V}
         #self.sims_imp = {w: get_similarity(w,self.target) * freq[w] for w in V}
         # simulated priors regarding t for each other player; if the other player is an agent as well, these don't matter
         self.t_priors = {p: np.array([freq.values()]) for p in players if not p==self.player}
         # (simulated) priors regarding i for each player; if a player is an imposter, these don't matter.
         self.i_priors = {p1: {p2: 0.5 for p2 in players if not p2==p1} for p1 in players}

    def get_target(self):
        mv = max(freq.values())
        res = [k for k in freq if freq[k] == mv]
        return(rd.choice(res))
    """
    def L0(self, u, p_player = [2/3, 1/3]):
        #sims = {w: get_similarity(w,self.target) * freq[w] for w in V}
        #p_imposter = get_similarity(u,self.target_imp) * p_player[1]
        #p_target_norm = self.sims[self.target]/sum([self.sims[w] for w in V])
        sim_sum = sum(self.sims.values())
        p_dic = {w: self.sims[w]/sim_sum for w in V}

        sims_imposter = {w: get_similarity(w,self.target_imp) * freq[w] for w in V}
        sim_sum_imp = sum(sims_imposter.values())
        p_dic_imp = {w: self.sims[w]/sim_sum for w in V}

        #p_agent_norm = p_agent/(p_agent + p_imposter)
        #p_imposter_norm = p_imposter/(p_agent + p_imposter)
        return((p_dic, p_dic_imp))

    """
    
    def L0(self, r, s, l, priors):
        # reasoner, speaker, listener
        # listeners sim_table if he's an imposter and doesn't know T 
        sim_table_l = get_sim_table(u, max(self.t_priors[l]))
        p_t = (1 - self.i_priors[r][l]) * sim_table * self.i_priors[l][s] + self.i_priors[r][l] * sim_table_l * 
        p_t_norm = p_t/sum(p_t)
        # very important part - does target always come first in the vector dimensions? If so, at which dimension do we need to mulitply? T
        # and over which do we need to normalize? 
        return(p_t_norm)

    """
    def S1(self):
        p_distr = []
        for u in V: 
            # in the future this might have to iterate over multiple listeners 
            # interestingly though, the agents don't care how the imposter perceives them, so this would only be a problem for the imposter
            l0_agent, l0_imp = self.L0(u)
            C = l0_imp[self.target]
            p_u = np.exp(self.alpha * (np.log(l0_agent[self.target])- C[u]))
            #print("p_u:", p_u)
            p_distr.append(p_u)

        p_sum = sum(p_distr)
        u_dic = {u: p/p_sum for u,p in zip(V,p_distr)}
        #print(p_distr)
        #print(u_dic)
        return(u_dic)
        """
    
    def S1(self): 
        player_p = {p: self.L0(self.player_priors[p]) for p in self.player_priors.keys()}
        """für U_S1(u,w) prop ln(P_L0(w|u)) – C(u)
        und P_S1(u|w) prop exp(alpha x U_S1(u,w))"""
        # might need some parameter to multiply cost by 
        # cost is just the L0 of the other player
        # do they reason about the beliefs of the others about the target or about the actual word state
        c_u = {p: player_p[[op for op in player_p if not op==p][0]] for p in player_p.keys()}
        u_s1 = {p: np.log(player_p[p]) - c_u[p] for p in player_p.keys()}


    def choose_u(self): 
        P_u = self.S1()
        mv = max(P_u.values())
        res = [k for k in P_u if P_u[k] == mv]
        return(rd.choice(res))
    
    def L1(self, u, player):
        p_agent = self.S1()[u] * self.P_w[player][0]
        p_imposter = self.S1()[u] * self.P_w[player][1]
        p_sum = sum(p_agent, p_imposter)
        p_agent_norm = p_agent/p_sum
        p_imposter_norm = p_imposter/p_sum
        # is the new probability distribution a mix of the old distribution and the newly calculated one or is it just the new one? 
        self.P_w[player] = [p_agent_norm, p_imposter_norm]
        return((p_agent_norm, p_imposter_norm))


class Imposter:
    def __init__(self, player): 
         self.player = player
         self.role = roles[player]
         self.target = self.get_target()
         self.P_t = freq.copy()
         self.alpha = rationality[player]
         self.target_imp = self.get_target()

    def get_target(self):
        mv = max(freq.values())
        res = [k for k in freq if freq[k] == mv]
        return(rd.choice(res))
    
    
    def L0(self, u, t, p_player = [2/3, 1/3]):
        p_agent = get_similarity(u,t) * p_player[0]
        p_imposter = get_similarity(u,self.target_imp) * p_player[1]
        p_agent_norm = p_agent/(p_agent + p_imposter)
        p_imposter_norm = p_imposter/(p_agent + p_imposter)
        return((p_agent_norm, p_imposter_norm))

    def S1(self, t):
        p_distr = []
        for u in V: 
            # what exactly is the world? The persona the player wants to communicate? That would always be agent.
            p_u = np.exp(self.alpha * (np.log(self.L0(u, t)[0])- C[u]))
            #print("p_u:", p_u)
            p_distr.append(p_u)

        p_sum = sum(p_distr)
        u_dic = {u: p/p_sum for u,p in zip(V,p_distr)}
        #print(p_distr)
        #print(u_dic)
        return(u_dic)

    def choose_u(self): 
        P_u = self.S1(self.target)
        mv = max(P_u.values())
        res = [k for k in P_u if P_u[k] == mv]
        return(rd.choice(res))
    

    def L1(self, u):
        t_distr = []
        for t in V: 
            p = self.S1(t)[u] * self.P_t[t]
            t_distr.append(p)

        p_sum = sum(t_distr)
        t_dic = {t: p/p_sum for t,p in zip(V,t_distr)}

        # is the new probability distribution a mix of the old distribution and the newly calculated one or is it just the new one? 
        self.P_t = t_dic

        return(t_dic)
    



class GameState:
    def __init__(self): 
        self.previous_words = []

    def say_word(self, word):
        self.previous_words += word



#

freq: {'Lt': 8.71e-06, 'Golf': 2.14e-05, 'Abteilung': 3.63e-05, 'There': 6.61e-06, 'Ergeben': 3.72e-05, 'Zelle': 1.38e-05, 'Linda': 7.94e-06, 'Vorsprung': 1.2e-05, 'Akte': 7.94e-06, 'Jungfrau': 1.02e-05}
target: ['Abteilung']


In [53]:
print(sim_table)

[[ 0.          1.          0.31959817  0.37949723  0.46415743  0.14474352
   0.36891884  0.40682372  0.22383884  0.07499813  0.24720225]
 [ 0.31959817  0.          1.          0.55250311  0.50798768  0.27516764
   0.53006846  0.51953804  0.52809376  0.57339746  0.53329408]
 [ 0.37949723  0.55250311  0.          1.          0.61974168  0.28051472
   0.60186076  0.51333499  0.64064693  0.37637678  0.69133103]
 [ 0.46415743  0.50798768  0.61974168  0.          1.00000012  0.20619601
   0.66384971  0.56526476  0.55248082  0.47846758  0.59067559]
 [ 0.14474352  0.27516764  0.28051472  0.20619601  0.          1.
   0.22074471 -0.01353713  0.36262673  0.05713809  0.32935715]
 [ 0.36891884  0.53006846  0.60186076  0.66384971  0.22074471  0.
   1.00000012  0.44434354  0.45618927  0.49631038  0.63376474]
 [ 0.40682372  0.51953804  0.51333499  0.56526476 -0.01353713  0.44434354
   0.          1.00000012  0.30903825  0.43347293  0.58383948]
 [ 0.22383884  0.52809376  0.64064693  0.55248082  0.3626

## next steps 

- implement cosine similarity 
- think about how to adjust probabilities with L1 functions 


In [50]:
p3 = Imposter("P3")

In [51]:
p3.L1(V[0])

/tmp/ipykernel_48459/2676187497.py:100: RuntimeWarning: invalid value encountered in log
  p_u = np.exp(self.alpha * (np.log(self.L0(u, t)[0])- C[u]))


{'Lt': np.float32(nan),
 'Golf': np.float32(nan),
 'Abteilung': np.float32(nan),
 'There': np.float32(nan),
 'Ergeben': np.float32(nan),
 'Zelle': np.float32(nan),
 'Linda': np.float32(nan),
 'Vorsprung': np.float32(nan),
 'Akte': np.float32(nan),
 'Jungfrau': np.float32(nan)}

In [33]:
with open('../output/random_nouns.json', 'r') as file:
    rd_100 = json.load(file)

print(rd_100)


['Ball', 'Künstler', 'Spaziergang', 'Partnerin', 'Bayer', 'Ereignis', 'Feuerwehr', 'Petition', 'Bundestag', 'Erfüllung', 'Ausweis', 'Rückkehr', 'Zitat', 'Verfasst', 'Schülerin', 'Wette', 'Aluminium', 'Saal', 'Einbezog', 'Eign', 'Klinik', 'Playstation', 'Nutze', 'Zufriedenheit', 'Opposition', 'Schweine', 'Messung', 'Steigerung', 'Hafen', 'Service', 'Zwischenzeit', 'Kenntnis', 'Rest', 'Terrorismus', 'Ulm', 'Wettbewerb', 'Freund', 'Element', 'Anwohner', 'Bgb', 'Empfänger', 'Rufe', 'Station', 'Milde', 'Gefasst', 'Invasion', 'Herauszufinde', 'Gewalt', 'Realschule', 'Solidarität', 'Produkt', 'Koffer', 'Mann', 'Verbrechen', 'Weisheit', 'Kontinent', 'Komplex', 'Pressemitteilung', 'Vorteil', 'Doktor', 'Symbol', 'Tasse', 'Scheidung', 'Stürmer', 'Statistik', 'Christ', 'Rundschau', 'Furcht', 'Dialog', 'Partie', 'Nation', 'Haus', 'Jungfrau', 'Tiger', 'Zieh', 'Rufe', 'Hochschule', 'Vorwurf', 'Rechtsprechung', 'Hotel', 'Fahrt', 'Wahrheit', 'Meinung', 'Verkehrt', 'Zwischenzeitlich', 'Einleitung', 'Nac